# Notebook 04 — Scratch CNN Training

**Architektur:** Vier Conv-Blöcke (mit BatchNorm) + Global Average Pooling + FC(256, 10)

```
Input: 3 × 224 × 224
Conv(3→32, 3×3, pad=1) + BN + ReLU + MaxPool(2)    → 32 × 112 × 112
Conv(32→64, 3×3, pad=1) + BN + ReLU + MaxPool(2)   → 64 × 56 × 56
Conv(64→128, 3×3, pad=1) + BN + ReLU + MaxPool(2)  → 128 × 28 × 28
Conv(128→256, 3×3, pad=1) + BN + ReLU + GAP(1)     → 256 × 1 × 1
Flatten → Dropout(p) → FC(256, 10)
```

**BatchNorm** nach jeder Conv-Schicht (Erweiterung über Kurs-Tutorial hinaus):
stabilisiert das Training, erlaubt höhere Lernraten, wirkt leicht regularisierend.

**Global Average Pooling** statt flatten+FC: reduziert Parameter-Bloat  
(12 544 → 256 Features), senkt Overfitting-Risiko bei ~1 800 Trainingsbildern.

**Hyperparameter** kommen aus Notebook 04b (HP-Suche). Hier auf beste Config setzen.

In [ ]:
import random
import sys
import time
from pathlib import Path

import matplotlib.pyplot as plt
import numpy as np
import torch
import torch.nn as nn
from torch.utils.data import DataLoader

ROOT = Path.cwd().parent
sys.path.insert(0, str(ROOT / "src"))
from dataset import AlbumCoverDataset, compute_class_weights  # noqa: E402

SPLITS_DIR      = ROOT / "data" / "splits"
CHECKPOINTS_DIR = ROOT / "data" / "checkpoints"
CHECKPOINTS_DIR.mkdir(parents=True, exist_ok=True)

# ── Reproduzierbarkeit ────────────────────────────────────────────────────
SEED = 42
random.seed(SEED)
np.random.seed(SEED)
torch.manual_seed(SEED)
torch.cuda.manual_seed_all(SEED)

# ── Device ────────────────────────────────────────────────────────────────
device = (
    torch.device("mps")  if torch.backends.mps.is_available()
    else torch.device("cuda") if torch.cuda.is_available()
    else torch.device("cpu")
)
print(f"Device: {device}")

## Konfiguration — beste HP aus Notebook 04b

In [ ]:
# ← Werte aus der Output-Zelle von 04b_hp_search.ipynb übernehmen.
BEST_LR      = 1e-3
BEST_DROPOUT = 0.5

EPOCHS        = 40
BATCH_SIZE    = 32
ES_PATIENCE   = 7   # Early-Stopping-Patience (Epochs ohne Val-Acc-Verbesserung)

print(f"LR={BEST_LR:.0e}  Dropout={BEST_DROPOUT}  Epochs={EPOCHS}  Batch={BATCH_SIZE}")

## DataLoaders & Class Weights

In [ ]:
pin         = device.type != "cpu"
num_workers = 2 if pin else 0

train_loader = DataLoader(
    AlbumCoverDataset(SPLITS_DIR / "train.csv", "train"),
    batch_size=BATCH_SIZE, shuffle=True,
    num_workers=num_workers, pin_memory=pin,
)
val_loader = DataLoader(
    AlbumCoverDataset(SPLITS_DIR / "val.csv", "val"),
    batch_size=BATCH_SIZE, shuffle=False,
    num_workers=num_workers, pin_memory=pin,
)

class_weights = compute_class_weights(SPLITS_DIR / "train.csv").to(device)
print(f"Train batches: {len(train_loader)}, Val batches: {len(val_loader)}")

## Modell

In [ ]:
class ScratchCNN(nn.Module):
    """Vier Conv-Blöcke (Conv + BN + ReLU + Pool) + GAP + Dropout + FC."""

    def __init__(self, dropout: float = 0.5) -> None:
        super().__init__()
        self.features = nn.Sequential(
            # Block 1
            nn.Conv2d(3,   32,  3, padding=1), nn.BatchNorm2d(32),  nn.ReLU(inplace=True), nn.MaxPool2d(2),
            # Block 2
            nn.Conv2d(32,  64,  3, padding=1), nn.BatchNorm2d(64),  nn.ReLU(inplace=True), nn.MaxPool2d(2),
            # Block 3
            nn.Conv2d(64,  128, 3, padding=1), nn.BatchNorm2d(128), nn.ReLU(inplace=True), nn.MaxPool2d(2),
            # Block 4 + GAP
            nn.Conv2d(128, 256, 3, padding=1), nn.BatchNorm2d(256), nn.ReLU(inplace=True),
            nn.AdaptiveAvgPool2d(1),
        )
        self.classifier = nn.Sequential(
            nn.Flatten(),
            nn.Dropout(p=dropout),
            nn.Linear(256, 10),
        )

    def forward(self, x: torch.Tensor) -> torch.Tensor:  # type: ignore[override]
        return self.classifier(self.features(x))


model     = ScratchCNN(dropout=BEST_DROPOUT).to(device)
optimizer = torch.optim.Adam(model.parameters(), lr=BEST_LR)
criterion = nn.CrossEntropyLoss(weight=class_weights)

total_params = sum(p.numel() for p in model.parameters())
print(f"Parameter gesamt: {total_params:,}")

## Training mit Early Stopping

In [ ]:
history = {
    "train_loss": [], "val_loss": [],
    "train_acc":  [], "val_acc":  [],
}

best_val_acc   = 0.0
patience_count = 0
checkpoint_path = CHECKPOINTS_DIR / "scratch_best.pt"
start_time = time.time()

for epoch in range(1, EPOCHS + 1):
    # ── Train ──
    model.train()
    t_loss, t_correct, t_total = 0.0, 0, 0
    for imgs, labels in train_loader:
        imgs, labels = imgs.to(device), labels.to(device)
        optimizer.zero_grad()
        out  = model(imgs)
        loss = criterion(out, labels)
        loss.backward()
        optimizer.step()
        t_loss    += loss.item() * len(labels)
        t_correct += (out.argmax(1) == labels).sum().item()
        t_total   += len(labels)

    # ── Val ──
    model.eval()
    v_loss, v_correct, v_total = 0.0, 0, 0
    with torch.no_grad():
        for imgs, labels in val_loader:
            imgs, labels = imgs.to(device), labels.to(device)
            out  = model(imgs)
            loss = criterion(out, labels)
            v_loss    += loss.item() * len(labels)
            v_correct += (out.argmax(1) == labels).sum().item()
            v_total   += len(labels)

    train_loss = t_loss / t_total
    train_acc  = t_correct / t_total
    val_loss   = v_loss / v_total
    val_acc    = v_correct / v_total

    history["train_loss"].append(train_loss)
    history["val_loss"].append(val_loss)
    history["train_acc"].append(train_acc)
    history["val_acc"].append(val_acc)

    print(f"Epoch {epoch:3d}/{EPOCHS}  "
          f"train_loss={train_loss:.4f} train_acc={train_acc:.3f}  "
          f"val_loss={val_loss:.4f} val_acc={val_acc:.3f}", end="")

    # ── Checkpoint & Early Stopping ──
    if val_acc > best_val_acc:
        best_val_acc = val_acc
        patience_count = 0
        torch.save(
            {"state_dict": model.state_dict(), "epoch": epoch, "val_acc": val_acc},
            checkpoint_path,
        )
        print("  ✓ checkpoint")
    else:
        patience_count += 1
        print(f"  (patience {patience_count}/{ES_PATIENCE})")
        if patience_count >= ES_PATIENCE:
            print(f"\nEarly Stopping nach Epoch {epoch}.")
            break

train_time = time.time() - start_time
print(f"\nBeste Val-Accuracy: {best_val_acc:.3f}")
print(f"Trainingszeit: {train_time:.0f}s")
print(f"Checkpoint: {checkpoint_path}")

## Loss- und Accuracy-Kurven

In [ ]:
epochs_run = range(1, len(history["train_loss"]) + 1)

fig, (ax1, ax2) = plt.subplots(1, 2, figsize=(12, 4))

ax1.plot(epochs_run, history["train_loss"], label="Train", color="steelblue")
ax1.plot(epochs_run, history["val_loss"],   label="Val",   color="darkorange")
ax1.set_xlabel("Epoch")
ax1.set_ylabel("Loss")
ax1.set_title("Scratch CNN — Loss")
ax1.legend()
ax1.grid(alpha=0.3)

ax2.plot(epochs_run, history["train_acc"], label="Train", color="steelblue")
ax2.plot(epochs_run, history["val_acc"],   label="Val",   color="darkorange")
ax2.set_xlabel("Epoch")
ax2.set_ylabel("Accuracy")
ax2.set_ylim(0, 1)
ax2.set_title(f"Scratch CNN — Accuracy  (best val={best_val_acc:.3f})")
ax2.legend()
ax2.grid(alpha=0.3)

plt.suptitle(f"Scratch CNN  |  LR={BEST_LR:.0e}, Dropout={BEST_DROPOUT}", fontsize=11)
plt.tight_layout()
plt.show()